In [6]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

In [7]:
!pip install -q -U torchao

## Phase 2: Persona Generator — Model Training

This notebook fine-tunes Qwen2.5-1.5B-Instruct using LoRA to generate
persona-conditioned dialogue for five characters finalized in Phase 1:
Jack (Fight Club), Bateman (American Psycho), Alvy (Annie Hall), Ben (The
Graduate), and Erin (Erin Brockovich). Training pairs (prompt, response,
persona_tag) were built in a separate data preparation notebook and
uploaded here as `training_pairs.csv`.

### Step 1: Load and format the training data

The training pairs (`training_pairs.csv`) are loaded and formatted using
the ChatML template expected by Qwen2.5-1.5B-Instruct. Each example tells
the model which character to respond as, shows the prompt it received,
and shows that character's actual line from the corpus as the target
completion. This teaches the model to generate replies that reflect each
character's distinct speech patterns rather than a generic response
style.

In [8]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("training_pairs.csv").dropna()
print(f"Loaded {len(df):,} training pairs")

def format_example(row):
    system = f"You are {row['persona_tag']}. Respond only in their voice, matching their vocabulary and speech patterns."
    text = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{row['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['response']}<|im_end|>"
    )
    return {"text": text}

formatted = df.apply(format_example, axis=1, result_type="expand")
dataset = Dataset.from_pandas(formatted)
print(dataset[0]["text"])

Loaded 1,434 training pairs
<|im_start|>system
You are bateman. Respond only in their voice, matching their vocabulary and speech patterns.<|im_end|>
<|im_start|>user
What? Oh, I'm...busy.<|im_end|>
<|im_start|>assistant
Listen, you're dating Luis, he's in Arizona. You're fucking me, and we haven't made plans. What could you possibly be up to tonight?<|im_end|>


### Step 2: Load the base model and attach a LoRA adapter

Qwen2.5-1.5B-Instruct is loaded and a LoRA adapter is attached to the
attention projection layers. LoRA freezes the original model weights and
trains only a small set of additional parameters, making fine-tuning
feasible on limited hardware while still adapting the model's behavior.

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


### Step 3: Configure and run training

The formatted dataset is tokenized and used to fine-tune the LoRA
adapter. Training runs for a small number of epochs given the limited
size of the dataset, using a cosine learning rate schedule and gradient
accumulation to simulate a larger effective batch size on limited GPU
memory. Only the LoRA adapter weights are updated during training, the
base model weights remain frozen.

In [10]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="generator_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

Map:   0%|          | 0/1434 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training...


Step,Training Loss
10,4.598507
20,2.794343
30,1.754412
40,1.615083
50,1.581565
60,1.526172
70,1.485993
80,1.562187
90,1.503797
100,1.468879


TrainOutput(global_step=270, training_loss=1.6349395363419144, metrics={'train_runtime': 3563.3801, 'train_samples_per_second': 1.207, 'train_steps_per_second': 0.076, 'total_flos': 8687361071775744.0, 'train_loss': 1.6349395363419144, 'epoch': 3.0})

### Step 4: Save the trained adapter

Once training completes, the LoRA adapter weights are saved separately
from the base model. This keeps the saved files small, since only the
adapter needs to be stored and later loaded alongside the base model for
inference.

In [11]:
ADAPTER_DIR = "generator_lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

Adapter saved to generator_lora_adapter


### Step 5: Test generation across all five characters

The fine-tuned model is used to generate candidate responses for the same
prompt across all five characters, as a check that training produced
distinct, in-character output for each persona rather than a single
generic style. This function, `generate_response()`, is also the actual
generation interface the project's reranking step will call.

In [12]:
def generate_response(prompt: str, persona_tag: str, n: int = 3, max_new_tokens: int = 60) -> list[str]:
    system = f"You are {persona_tag}. Respond only in their voice, matching their vocabulary and speech patterns."
    text = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        num_return_sequences=n,
        pad_token_id=tokenizer.eos_token_id,
    )
    responses = []
    for output in outputs:
        generated = output[inputs["input_ids"].shape[1]:]
        text_out = tokenizer.decode(generated, skip_special_tokens=True)
        responses.append(text_out.strip())
    return responses

for tag in ["jack", "bateman", "alvy", "ben", "erin"]:
    print(f"=== {tag.upper()} ===")
    results = generate_response("How's your day going?", tag, n=2)
    for i, r in enumerate(results, 1):
        print(f"  {i}: {r}")
    print()

=== JACK ===
  1: Good.  Just an ordinary day...
  2: I'm fine...  I mean, the weather is good.  There was a party at my office yesterday...  there wasn't anyone else to go.

=== BATEMAN ===
  1: Good, good. You know what? It was great. I mean, it had been a while since I've actually seen anyone. You get used to everybody you see on the street, or even at work...  but this is different. I don't recognize anybody. So far as I'm
  2: Great... you want to come with me down town for a couple hours?

=== ALVY ===
  1: I mean... it was great! It was so wonderful that I-I got up this morning and went down to the beach and uh- I saw a lot of people there. Oh god, what kind of an ass is this guy? What do you know about him? You must be from New York
  2: Oh, great! Good times for the future and now. I'm really excited 'bout this one thing.

=== BEN ===
  1: Not so bad -- well not as bad as yours I suppose but it's still pretty good.  It is better than it was. But then again you get used to thin

### Step 6: Package the adapter for download

The saved adapter folder is compressed into a single zip file so it can
be downloaded from Colab and later added to the project repository or
used locally for inference.

In [13]:
import shutil
shutil.make_archive("generator_lora_adapter", "zip", ADAPTER_DIR)
print("Zipped adapter ready for download: generator_lora_adapter.zip")

from google.colab import files
files.download("generator_lora_adapter.zip")

Zipped adapter ready for download: generator_lora_adapter.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>